In [ ]:
from pathlib import Path

# Find the repository root so these notebooks work on other computers.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "Dataset Specs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Dataset Specs").is_dir():
    raise FileNotFoundError("Start Jupyter from inside the Hospital Dataset project folder.")

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

data_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/90_Days"
)

arima_df = pd.read_csv(
    data_dir / "arima_occupancy_predictions.csv",
    parse_dates=["date"]
)

rf_df = pd.read_csv(
    data_dir / "random_forest_occupancy_predictions.csv",
    parse_dates=["date"]
)

gb_df = pd.read_csv(
    data_dir / "gradient_boosting_occupancy_predictions.csv",
    parse_dates=["date"]
)

print("ARIMA rows:", len(arima_df))
print("Random Forest rows:", len(rf_df))
print("Gradient Boosting rows:", len(gb_df))

In [ ]:
df_stacking = arima_df.merge(rf_df, on="date")
df_stacking = df_stacking.merge(gb_df, on="date")

rf_cols = [f"rf_{i}" for i in range(1, 6)]
gb_cols = [f"gb_{i}" for i in range(1, 6)]

df_stacking["rf_predicted_occupied_beds"] = (
    df_stacking[rf_cols].mean(axis=1)
)

df_stacking["gb_predicted_occupied_beds"] = (
    df_stacking[gb_cols].mean(axis=1)
)

print("Combined rows:", len(df_stacking))
print(df_stacking[
    [
        "date",
        "actual_occupied_beds",
        "arima_predicted_occupied_beds",
        "rf_predicted_occupied_beds",
        "gb_predicted_occupied_beds"
    ]
].head())

In [ ]:
# Select the three base-model predictions
X_meta = df_stacking[
    [
        "arima_predicted_occupied_beds",
        "rf_predicted_occupied_beds",
        "gb_predicted_occupied_beds"
    ]
].copy()

# Actual occupancy values
y_meta = df_stacking["actual_occupied_beds"].copy()

# Chronological 70/30 split
split_point = int(len(df_stacking) * 0.70)

X_train_meta = X_meta.iloc[:split_point]
X_test_meta = X_meta.iloc[split_point:]

y_train_meta = y_meta.iloc[:split_point]
y_test_meta = y_meta.iloc[split_point:]

print("Meta-training rows:", len(X_train_meta))
print("Meta-testing rows:", len(X_test_meta))

print(
    "Meta-training period:",
    df_stacking["date"].iloc[0].date(),
    "to",
    df_stacking["date"].iloc[split_point - 1].date()
)

print(
    "Meta-testing period:",
    df_stacking["date"].iloc[split_point].date(),
    "to",
    df_stacking["date"].iloc[-1].date()
)

In [ ]:
from sklearn.linear_model import Ridge

# Train the meta-model
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_meta, y_train_meta)

# Predict occupancy on the unseen meta-testing period
y_pred_stacking = ridge_model.predict(X_test_meta)

print("Ridge stacking model trained successfully!")
print("Number of test predictions:", len(y_pred_stacking))

print("\nLearned model coefficients:")
for name, coefficient in zip(X_meta.columns, ridge_model.coef_):
    print(f"{name}: {coefficient:.4f}")

print("Intercept:", round(ridge_model.intercept_, 4))

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

y_true = y_test_meta
y_pred = y_pred_stacking

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("Ridge Stacking Performance — 90-Day Experiment")
print("Meta-testing period: 2024-12-04 to 2024-12-31")
print(f"RMSE: {rmse:.3f} beds")
print(f"MAE:  {mae:.3f} beds")
print(f"MAPE: {mape:.3f}%")

In [ ]:
from pathlib import Path
import pandas as pd

output_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/90_Days"
)
output_dir.mkdir(parents=True, exist_ok=True)

# Save stacking metrics
stacking_metrics_df = pd.DataFrame([{
    "Model": "Ridge Stacking",
    "RMSE": rmse,
    "MAE": mae,
    "MAPE (%)": mape
}])

stacking_metrics_df.to_csv(
    output_dir / "ridge_stacking_metrics.csv",
    index=False
)

# Save predictions with dates and actual values
stacking_predictions_df = pd.DataFrame({
    "date": df_stacking["date"].iloc[split_point:].to_numpy(),
    "actual_occupied_beds": y_test_meta.to_numpy(),
    "ridge_stacking_predicted_occupied_beds": y_pred_stacking
})

stacking_predictions_df.to_csv(
    output_dir / "ridge_stacking_predictions.csv",
    index=False
)

print("Stacking metrics and predictions saved successfully.")
print("Metrics:", output_dir / "ridge_stacking_metrics.csv")
print("Predictions:", output_dir / "ridge_stacking_predictions.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

data_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/90_Days"
)

arima_df = pd.read_csv(
    data_dir / "arima_occupancy_predictions.csv",
    parse_dates=["date"]
)

rf_df = pd.read_csv(
    data_dir / "random_forest_occupancy_predictions.csv",
    parse_dates=["date"]
)

gb_df = pd.read_csv(
    data_dir / "gradient_boosting_occupancy_predictions.csv",
    parse_dates=["date"]
)

df_fusion = arima_df.merge(rf_df, on="date")
df_fusion = df_fusion.merge(gb_df, on="date")

# Average the five predictions from each machine-learning model
rf_cols = [f"rf_{i}" for i in range(1, 6)]
gb_cols = [f"gb_{i}" for i in range(1, 6)]

df_fusion["rf_predicted_occupied_beds"] = (
    df_fusion[rf_cols].mean(axis=1)
)

df_fusion["gb_predicted_occupied_beds"] = (
    df_fusion[gb_cols].mean(axis=1)
)

# Simple average of the three base models
df_fusion["fusion_predicted_occupied_beds"] = df_fusion[
    [
        "arima_predicted_occupied_beds",
        "rf_predicted_occupied_beds",
        "gb_predicted_occupied_beds"
    ]
].mean(axis=1)

print("Combined rows:", len(df_fusion))
print("Fusion dataframe recreated successfully.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Get the same 28 dates used to test Ridge Stacking
comparison_dates = df_stacking["date"].iloc[split_point:]

comparison_df = df_fusion[
    df_fusion["date"].isin(comparison_dates)
].copy()

y_true = comparison_df["actual_occupied_beds"]

models = {
    "ARIMA": comparison_df["arima_predicted_occupied_beds"],
    "Random Forest": comparison_df["rf_predicted_occupied_beds"],
    "Gradient Boosting": comparison_df["gb_predicted_occupied_beds"],
    "Simple Fusion Average": comparison_df["fusion_predicted_occupied_beds"],
    "Ridge Stacking": y_pred_stacking
}

results = []

for model_name, predictions in models.items():
    rmse = np.sqrt(mean_squared_error(y_true, predictions))
    mae = mean_absolute_error(y_true, predictions)
    mape = np.mean(np.abs((y_true - predictions) / y_true)) * 100

    results.append({
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape
    })

comparison_metrics_df = (
    pd.DataFrame(results)
    .sort_values("RMSE")
    .reset_index(drop=True)
    .round(3)
)

print("Model comparison — same 28 test dates")
print("Period: 2024-12-04 to 2024-12-31")
print(comparison_metrics_df.to_string(index=False))